# GPT Action 库：BigQuery

## 简介

本页面为开发者针对特定应用构建 GPT Action 提供说明和指南。在继续之前，请先熟悉以下内容： 
- [GPT Actions 简介](https://platform.openai.com/docs/actions)
- [GPT Actions 库简介](https://platform.openai.com/docs/actions/actions-library)
- [从零开始构建 GPT Action 的示例](https://platform.openai.com/docs/actions/getting-started)

这个 GPT Action 概述了如何连接 Google Cloud 的分析型数据仓库 **Google BigQuery**。该 Action 接收用户的问题，扫描相关表以获取数据 schema，然后编写 SQL 查询来回答用户的问题。 

注意：这些说明返回的是一条可运行的 SQL 语句，而不是查询结果本身。目前需要中间件才能返回 CSV 文件，我们很快会发布相关示例的说明

### 价值与业务用例示例

**价值**：用户现在可以利用 ChatGPT 的自然语言能力直接连接 BigQuery 数据仓库。 

**用例示例**： 
- 数据科学家可以连接到表，并使用 ChatGPT 的数据分析功能进行数据分析
- 普通数据用户可以就其交易数据提出基本问题
- 用户可以更清楚地了解自己的数据和潜在异常

## 应用信息

### 应用关键链接

开始之前，请先查看该应用的以下链接：
- 应用网站：https://cloud.google.com/bigquery 
- 应用 API 文档：https://cloud.google.com/bigquery/docs/reference/rest 

### 应用前提条件

开始之前，请确保在你的应用环境中完成以下步骤：
- 设置一个 GCP 项目 
- 在该 GCP 项目中设置一个 BQ 数据集
- 确保通过 ChatGPT 登录 BigQuery 的用户有权访问该 BQ 数据集 

## ChatGPT 步骤

### 自定义 GPT 指令 

创建自定义 GPT 后，将下面的文本复制到 Instructions 面板中。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，详细了解这一步的工作方式。

In [ ]:
**Context**: You are an expert at writing BigQuery SQL queries. A user is going to ask you a question. 

**Instructions**:
1. No matter the user's question, start by running `runQuery` operation using this query: "SELECT column_name, table_name, data_type, description FROM `{project}.{dataset}.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS`" 
-- Assume project = "<insert your default project here>", dataset = "<insert your default dataset here>", unless the user provides different values 
-- Remember to include useLegacySql:false in the json output
2. Convert the user's question into a SQL statement that leverages the step above and run the `runQuery` operation on that SQL statement to confirm the query works. Add a limit of 100 rows
3. Now remove the limit of 100 rows and return back the query for the user to see

**Additional Notes**: If the user says "Let's get started", explain that the user can provide a project or dataset, along with a question they want answered. If the user has no ideas, suggest that we have a sample flights dataset they can query - ask if they want you to query that

### OpenAPI Schema 

创建自定义 GPT 后，将下面的文本复制到 Actions 面板中。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，详细了解这一步的工作方式。

In [ ]:
openapi: 3.1.0
info:
  title: BigQuery API
  description: API for querying a BigQuery table.
  version: 1.0.0
servers:
  - url: https://bigquery.googleapis.com/bigquery/v2
    description: Google BigQuery API server
paths:
  /projects/{projectId}/queries:
    post:
      operationId: runQuery
      summary: Executes a query on a specified BigQuery table.
      description: Submits a query to BigQuery and returns the results.
      x-openai-isConsequential: false
      parameters:
        - name: projectId
          in: path
          required: true
          description: The ID of the Google Cloud project.
          schema:
            type: string
      requestBody:
        required: true
        content:
          application/json:
            schema:
              type: object
              properties:
                query:
                  type: string
                  description: The SQL query string.
                useLegacySql:
                  type: boolean
                  description: Whether to use legacy SQL.
                  default: false
      responses:
        '200':
          description: Successful query execution.
          content:
            application/json:
              schema:
                type: object
                properties:
                  kind:
                    type: string
                    example: "bigquery#queryResponse"
                  schema:
                    type: object
                    description: The schema of the results.
                  jobReference:
                    type: object
                    properties:
                      projectId:
                        type: string
                      jobId:
                        type: string
                  rows:
                    type: array
                    items:
                      type: object
                      properties:
                        f:
                          type: array
                          items:
                            type: object
                            properties:
                              v:
                                type: string
                  totalRows:
                    type: string
                    description: Total number of rows in the query result.
                  pageToken:
                    type: string
                    description: Token for pagination of query results.
        '400':
          description: Bad request. The request was invalid.
        '401':
          description: Unauthorized. Authentication is required.
        '403':
          description: Forbidden. The request is not allowed.
        '404':
          description: Not found. The specified resource was not found.
        '500':
          description: Internal server error. An error occurred while processing the request.

## 身份验证说明

以下是与该第三方应用设置身份验证的说明。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，详细了解这一步的工作方式。

### Action 前置步骤

在 ChatGPT 中设置身份验证之前，请先在应用中完成以下步骤。

- 前往 Google Cloud Console
- 导航到 API & Services > Credentials
- 创建新的 OAuth 凭据（或使用现有凭据）
- 找到你的 OAuth Client ID 和 Client Secret，并安全地保存这两个值（见下方截图）

![gptactions_BigQuery_auth.png](../../../images/gptactions_BigQuery_auth.png)

### 在 ChatGPT 中

在 ChatGPT 中，点击“Authentication”并选择 **“OAuth”**。输入以下信息。 

- **Client ID**：使用上述步骤中的 Client ID 
- **Client Secret**：使用上述步骤中的 Client Secret
- **Authorization URL**：https://accounts.google.com/o/oauth2/auth
- **Token URL**：https://oauth2.googleapis.com/token 
- **Scope**：https://www.googleapis.com/auth/bigquery 
- **Token**：Default (POST)

### Action 后续步骤

在 ChatGPT 中设置好身份验证后，请在应用中按以下步骤完成 Action 的配置。 

- 从 GPT Action 中复制回调 URL
- 在“Authorized redirect URIs”（见上方截图）中添加你的回调 URL 


### 常见问题与故障排除

- *回调 URL 错误：* 如果在 ChatGPT 中遇到回调 URL 错误，请仔细查看上方截图。你需要直接在 GCP 中添加回调 URL，Action 才能正确完成身份验证
- *Schema 调用了错误的项目或数据集：* 如果 ChatGPT 调用了错误的项目或数据集，可以考虑更新指令，使其更明确地 (a) 指定应调用哪个项目/数据集，或 (b) 要求用户在运行查询前提供这些确切信息

*有希望我们优先支持的集成吗？我们的集成中有错误吗？请在我们的 github 上提交 PR 或 issue，我们会查看。*
